In [1]:
import polars as pl

from datetime import date

from ohlc_dss_model.data.data_loader import load_parquet
from ohlc_dss_model.data.integrity import sort_data
from ohlc_dss_model.data.tagging import intraday_session_tagging, session_tagging

from ohlc_dss_model.utils.dt_utils import convert_to_timezone


In [2]:
df = load_parquet()
df = convert_to_timezone(df)
df = sort_data(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = df.filter(
    (pl.col("Session") != date(2016, 11, 15)), (pl.col("Session") != date(2025, 10, 1))
)

print(df.head(5))

shape: (5, 9)
┌────────────────┬────────┬────────┬────────┬───┬────────┬────────────┬────────────┬───────────────┐
│ DateTime       ┆ Open   ┆ High   ┆ Low    ┆ … ┆ Volume ┆ TickVolume ┆ Session    ┆ Intraday_Sess │
│ ---            ┆ ---    ┆ ---    ┆ ---    ┆   ┆ ---    ┆ ---        ┆ ---        ┆ ion           │
│ datetime[μs,   ┆ f64    ┆ f64    ┆ f64    ┆   ┆ i64    ┆ i64        ┆ date       ┆ ---           │
│ America/New_Yo ┆        ┆        ┆        ┆   ┆        ┆            ┆            ┆ str           │
│ rk]            ┆        ┆        ┆        ┆   ┆        ┆            ┆            ┆               │
╞════════════════╪════════╪════════╪════════╪═══╪════════╪════════════╪════════════╪═══════════════╡
│ 2016-11-15     ┆ 4765.7 ┆ 4770.5 ┆ 4765.7 ┆ … ┆ 0      ┆ 262        ┆ 2016-11-16 ┆ Asia          │
│ 18:00:00 EST   ┆        ┆        ┆        ┆   ┆        ┆            ┆            ┆               │
│ 2016-11-15     ┆ 4769.9 ┆ 4772.7 ┆ 4768.6 ┆ … ┆ 0      ┆ 259        ┆ 2016-

We need to group our data by its session such that each row represents one trading day